![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 05: NLP for Clinical Text
***
**Learning objectives**
- Understand BERT bidirectional attention architecture
- Contrast general BERT vs BioBERT vs ClinicalBERT pre-training
- Extract sentence embeddings using `sentence-transformers`
- Build a cosine similarity matrix across clinical snippets
- Perform zero-shot clinical text classification
- Write a fine-tuning starter template for ClinicalBERT

**Estimated time:** 3 hours | **Level:** Advanced | **Libraries:** `sentence-transformers`, `transformers`, `sklearn`


## 1. Setup

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import classification_report
warnings.filterwarnings("ignore")
os.makedirs("/tmp/mod05", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42)

TRANSFORMERS_OK       = False
SENTENCE_TRANS_OK     = False

try:
    import torch
    from transformers import AutoTokenizer, AutoModel, pipeline
    TRANSFORMERS_OK = True
    print(f"transformers available | torch {torch.__version__}")
except ImportError:
    print("pip install transformers torch")

try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_TRANS_OK = True
    print("sentence-transformers available")
except ImportError:
    print("pip install sentence-transformers")


## 2. BERT Architecture Overview

```
Input:   [CLS] patient has heart failure [SEP]
            ↓  WordPiece tokenisation
Token IDs + Attention Mask + Position IDs
            ↓
12 × Transformer Encoder layers (BERT-base)
  - Multi-head Self-Attention (bidirectional — sees ALL tokens)
  - Feed-Forward + LayerNorm + Residual
            ↓
768-dim contextual embedding per token
[CLS] embedding → sentence representation for classification
```

| Model | Pre-trained on | Best use case |
|---|---|---|
| `bert-base-uncased` | Wikipedia + BookCorpus | General NLP baseline |
| `dmis-lab/biobert-v1.1` | PubMed + PMC full-text | Biomedical literature |
| `emilyalsentzer/Bio_ClinicalBERT` | MIMIC-III clinical notes | EHR / discharge summaries |
| `microsoft/BiomedNLP-PubMedBERT-base` | PubMed only (from scratch) | Biomedical NLP tasks |


In [ ]:
CLINICAL_SNIPPETS = [
    {"text":"Patient has acute decompensated heart failure, EF 30%, BNP 1450, bilateral pitting edema requiring IV diuresis.",
     "label":"Cardiac","readmitted":1},
    {"text":"STEMI confirmed by EKG, troponin 2.4, activating cath lab for emergency PCI, aspirin and heparin given.",
     "label":"Cardiac","readmitted":1},
    {"text":"Atrial fibrillation with rapid ventricular response, rate controlled with metoprolol, anticoagulation started.",
     "label":"Cardiac","readmitted":0},
    {"text":"COPD exacerbation with diffuse wheezes, albuterol nebs, methylprednisolone, and azithromycin started.",
     "label":"Pulmonary","readmitted":0},
    {"text":"Community-acquired pneumonia, right lower lobe consolidation on CXR, ceftriaxone and azithromycin.",
     "label":"Pulmonary","readmitted":0},
    {"text":"Bilateral pulmonary embolism on CT angiography, RV strain, initiating anticoagulation with rivaroxaban.",
     "label":"Pulmonary","readmitted":1},
    {"text":"Diabetic ketoacidosis, glucose 520, anion gap 24, insulin drip and aggressive IV fluid resuscitation.",
     "label":"Diabetes","readmitted":1},
    {"text":"Uncontrolled T2DM, HbA1c 11.2, intensifying regimen with GLP-1 agonist and SGLT2 inhibitor.",
     "label":"Diabetes","readmitted":0},
    {"text":"Acute kidney injury, creatinine 3.8 from baseline 1.0, holding nephrotoxins, nephrology consulted.",
     "label":"Renal","readmitted":1},
    {"text":"ESRD on hemodialysis, hyperkalemia K 6.4, emergent dialysis performed, access fistula patent.",
     "label":"Renal","readmitted":1},
    {"text":"Septic shock, BP 78/42, norepinephrine initiated, broad-spectrum antibiotics, source investigation.",
     "label":"Sepsis","readmitted":1},
    {"text":"Bacteremia with MSSA, blood cultures positive, transitioning to oxacillin, 6-week course planned.",
     "label":"Sepsis","readmitted":0},
]
df_bert = pd.DataFrame(CLINICAL_SNIPPETS)
print(f"BERT corpus: {len(df_bert)} clinical snippets")
print(df_bert.groupby("label")["readmitted"].value_counts())


## 3. Sentence Embeddings

In [ ]:
if SENTENCE_TRANS_OK:
    print("Loading sentence-transformers (all-MiniLM-L6-v2)...")
    print("For clinical work, prefer: pritamdeka/PubMedBERT-mnli-snli-scinli-scitail-mednli-stsb")
    sbert = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = sbert.encode(df_bert["text"].tolist(), show_progress_bar=False)
    print(f"Embeddings shape: {embeddings.shape}")
    EMBED_OK = True
else:
    # Fallback: TF-IDF + SVD pseudo-embeddings
    print("sentence-transformers not available — using TF-IDF + SVD fallback")
    tfidf = TfidfVectorizer(max_features=500)
    X_sp  = tfidf.fit_transform(df_bert["text"])
    svd   = TruncatedSVD(n_components=64, random_state=42)
    embeddings = svd.fit_transform(X_sp)
    print(f"Fallback embeddings shape: {embeddings.shape}")
    EMBED_OK = True
    sbert = None


In [ ]:
# Cosine similarity matrix
sim_matrix = cosine_similarity(embeddings)
labels_short = [f"{row.label[:4]}_{i}" for i,(_,row) in enumerate(df_bert.iterrows())]

fig, ax = plt.subplots(figsize=(9,7))
im = ax.imshow(sim_matrix, cmap="Blues", vmin=0.3, vmax=1.0)
ax.set_xticks(range(len(labels_short))); ax.set_xticklabels(labels_short, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(labels_short))); ax.set_yticklabels(labels_short, fontsize=8)
plt.colorbar(im, ax=ax, label="Cosine similarity")
ax.set_title("Clinical Embedding Cosine Similarity Matrix\n(Same-condition pairs expected to be more similar)")
for i in range(len(embeddings)):
    for j in range(len(embeddings)):
        ax.text(j,i,f"{sim_matrix[i,j]:.2f}",ha="center",va="center",fontsize=6,
                color="white" if sim_matrix[i,j]>0.75 else "black")
plt.tight_layout()
plt.savefig("/tmp/mod05/bert_similarity.png",bbox_inches="tight"); plt.show()


## 4. PCA Projection of Embedding Space

In [ ]:
pca = PCA(n_components=2, random_state=42)
emb_2d = pca.fit_transform(embeddings)

COLORS = {"Cardiac":"#D65F5F","Pulmonary":"#4878CF","Diabetes":"#6ACC65",
          "Renal":"#B47CC7","Sepsis":"#D4A017"}
shapes = {0:"o", 1:"*"}

fig, ax = plt.subplots(figsize=(10,7))
for i, (_, row) in enumerate(df_bert.iterrows()):
    color  = COLORS[row.label]
    marker = shapes[row.readmitted]
    ax.scatter(emb_2d[i,0], emb_2d[i,1], c=color, marker=marker,
               s=160 if row.readmitted else 80, alpha=0.85, zorder=3,
               edgecolors="white", linewidth=0.5)
    ax.annotate(row.label[:4], (emb_2d[i,0]+0.02, emb_2d[i,1]+0.02), fontsize=7)

legend_cond  = [mpatches.Patch(color=c,label=l) for l,c in COLORS.items()]
legend_shape = [plt.Line2D([0],[0],marker="*",color="gray",ms=12,ls="",label="Readmitted"),
                plt.Line2D([0],[0],marker="o",color="gray",ms=8, ls="",label="Not readmitted")]
ax.legend(handles=legend_cond+legend_shape, fontsize=9, loc="upper right")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
ax.set_title("PCA of Clinical BERT Embeddings (star=readmitted)")
plt.tight_layout()
plt.savefig("/tmp/mod05/bert_pca.png",bbox_inches="tight"); plt.show()


## 5. Zero-Shot Classification via Semantic Similarity

In [ ]:
CANDIDATE_DESCRIPTIONS = [
    "This patient has a cardiac or heart condition such as heart failure or myocardial infarction",
    "This patient has a pulmonary or lung condition such as COPD pneumonia or pulmonary embolism",
    "This patient has diabetes or a metabolic condition such as DKA or hypoglycemia",
    "This patient has a kidney or renal condition such as acute kidney injury or ESRD",
    "This patient has an infection or sepsis condition such as bacteremia or septic shock",
]
CONDITIONS_ZS = ["Cardiac","Pulmonary","Diabetes","Renal","Sepsis"]

if SENTENCE_TRANS_OK and sbert is not None:
    label_embs = sbert.encode(CANDIDATE_DESCRIPTIONS)
    scores_zs  = cosine_similarity(embeddings, label_embs)
    preds = [CONDITIONS_ZS[s.argmax()] for s in scores_zs]
    trues = df_bert["label"].tolist()
    correct = sum(p==t for p,t in zip(preds,trues))
    print(f"Zero-shot accuracy: {correct}/{len(trues)} ({correct/len(trues)*100:.0f}%)\n")
    print(f"{'True':12s} {'Predicted':12s} {'Score':8s} Match")
    print("-"*45)
    for true,pred,score_row in zip(trues,preds,scores_zs):
        top_score = score_row.max()
        match = "✓" if true==pred else "✗"
        print(f"  {true:12s} {pred:12s} {top_score:.3f}   {match}")
else:
    # TF-IDF fallback zero-shot
    tfidf_zs = TfidfVectorizer(ngram_range=(1,2))
    all_texts = CANDIDATE_DESCRIPTIONS + df_bert["text"].tolist()
    X_zs = tfidf_zs.fit_transform(all_texts)
    label_vecs = X_zs[:len(CANDIDATE_DESCRIPTIONS)]
    note_vecs  = X_zs[len(CANDIDATE_DESCRIPTIONS):]
    scores_zs  = cosine_similarity(note_vecs, label_vecs)
    preds = [CONDITIONS_ZS[s.argmax()] for s in scores_zs]
    trues = df_bert["label"].tolist()
    correct = sum(p==t for p,t in zip(preds,trues))
    print(f"Fallback TF-IDF zero-shot accuracy: {correct}/{len(trues)} ({correct/len(trues)*100:.0f}%)")


## 6. Fine-Tuning Starter Template

In [ ]:
# ── ClinicalBERT fine-tuning template (run on GPU with real data) ─────────────
FINE_TUNE_TEMPLATE = ''' from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer)  MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT" tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME) model      = AutoModelForSequenceClassification.from_pretrained( MODEL_NAME, num_labels=2, id2label={0:"Not readmitted", 1:"Readmitted"} )  def tokenize_fn(examples): return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)  training_args = TrainingArguments( output_dir             = "./clinical_bert_readmission", num_train_epochs       = 3, per_device_train_batch_size = 16, per_device_eval_batch_size  = 32, warmup_steps           = 200, weight_decay           = 0.01, learning_rate          = 2e-5, evaluation_strategy    = "epoch", load_best_model_at_end = True, metric_for_best_model  = "eval_roc_auc", fp16                   = True, report_to              = "none", ) # trainer = Trainer(model=model, args=training_args, #                   train_dataset=train_tok, eval_dataset=val_tok, #                   compute_metrics=compute_metrics) # trainer.train() '''
print("Fine-tuning key parameters:")
print("  Learning rate  : 2e-5 (BERT needs smaller LR than training from scratch)")
print("  Batch size     : 16-32 (GPU memory dependent)")
print("  Epochs         : 3-5 (clinical text overfits quickly)")
print("  Max token len  : 512 (~350 words; long notes must be truncated or chunked)")
print("  Warmup steps   : 10% of total training steps")
print()
print("Model recommendations by task:")
print("  Discharge summaries  : emilyalsentzer/Bio_ClinicalBERT (MIMIC-III)")
print("  Biomedical journals  : dmis-lab/biobert-v1.1 (PubMed + PMC)")
print("  General biomedical   : microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract")


## 7. Knowledge Check
1. Why is ClinicalBERT expected to outperform general BERT on EHR text?
2. Zero-shot score 0.73 for a cardiac note — is this high confidence? How do you calibrate?
3. The PCA shows Cardiac and Pulmonary clusters intermixed — what does this reflect clinically?
4. BERT max length is 512 tokens. A discharge summary is 800 tokens. What are your options?
5. Compare cosine similarity between two Cardiac snippets vs one Cardiac + one Sepsis snippet.

In [ ]:
if SENTENCE_TRANS_OK and sbert is not None:
    cardiac_texts = df_bert[df_bert.label=="Cardiac"]["text"].tolist()[:2]
    sepsis_text   = df_bert[df_bert.label=="Sepsis"]["text"].iloc[0]
    embs3 = sbert.encode(cardiac_texts + [sepsis_text])
    sim_cc = cosine_similarity([embs3[0]],[embs3[1]])[0][0]
    sim_cs = cosine_similarity([embs3[0]],[embs3[2]])[0][0]
    print(f"Cardiac vs Cardiac : {sim_cc:.4f}")
    print(f"Cardiac vs Sepsis  : {sim_cs:.4f}")
    print(f"Within-condition similarity higher by: {sim_cc-sim_cs:+.4f}")
else:
    print("sentence-transformers not available — install to run this exercise")
    print("Expected: within-condition similarity > across-condition similarity")


***
**Next → NB-06: ICD Coding & Medication Extraction**
